# Conditional Probability Classifiers

> This module contains code to build a conditional probability classifier, which is inspired by this paper: [https://arxiv.org/pdf/1911.06475.pdf](https://arxiv.org/pdf/1911.06475.pdf)

In [ ]:
#| default_exp models.conditional_prob_classifiers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from collections import defaultdict
import numpy as np
import torch
from transformers.models.roberta.configuration_roberta import RobertaConfig
from transformers.models.roberta.modeling_roberta import RobertaModel
from that_nlp_library.models.classifiers import RobertaConcatHeadSimple
from sklearn.preprocessing import MultiLabelBinarizer
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel
from that_nlp_library.models.classifiers import loss_for_classification

In [ ]:
#| export
def build_standard_condition_mask(df,label1_encoder,label2_encoder,label1,label2):
    L2_SIZE = len(label2_encoder.classes_)
    L1_SIZE = len(label1_encoder.classes_)
    
    l12idx = {v:i for i,v in enumerate(label1_encoder.classes_)}
    l22idx = {v:i for i,v in enumerate(label2_encoder.classes_)}
    df_labels = df[[label1,label2]].drop_duplicates().sort_values([label1,label2])
    df_labels = df_labels.groupby([label1])[label2].apply('|'.join).reset_index()
    
    _d = defaultdict(list)
    for i,j in df_labels.values:
        for k in j.split('|'):
            k = k.strip()
            _d[l12idx[i]].append(l22idx[k])
    
    mask_l1 = torch.eye(L1_SIZE) ==1
    
    mlb = MultiLabelBinarizer()
    mlb.fit([np.arange(L2_SIZE)])
    mask_l2 = torch.tensor(mlb.transform(list(_d.values())) == 1)
    
    mask_final= torch.cat((mask_l1,mask_l2),1)
    
    return mask_final

In [ ]:
#| export
class RobertaHSCCProbSequenceClassification(RobertaPreTrainedModel):
    """
    Roberta Conditional Probability Architecture with Hidden-State-Concatenation for Sequence Classification task
    """
    config_class = RobertaConfig

    def __init__(self, 
                 config, # HuggingFace model configuration
                 pretrained_roberta=None, # HuggingFace Roberta Body (useful for knowledge transfering task)
                 concathead_class=RobertaConcatHeadSimple,
                 classifier_dropout=0.1, # Dropout ratio (for dropout layer right before the last nn.Linear)
                 last_hidden_size=768, # Last hidden size (before the last nn.Linear)
                 size_l1=None, # Number of classes for head 1
                 size_l2=None, # Number of classes for head 2
                 standard_mask=None, # Mask for conditional probability
                 device=None # CPU or GPU
                ):
        super().__init__(config)
        self.device = device if device is not None else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.size_l1 = size_l1
        self.size_l2 = size_l2
        
        # set num_labels for config
        num_labels = size_l1+size_l2
        config.num_labels = num_labels
        
        self.roberta = RobertaModel(config, add_pooling_layer=False) if pretrained_roberta is None else pretrained_roberta
        self.standard_mask = standard_mask
        self.classification_head = concathead_class(config=config,classifier_dropout=classifier_dropout,
                                                    last_hidden_size=last_hidden_size,
                                                    n_output=num_labels) 

        if pretrained_roberta is None:
            self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                labels=None, **kwargs):
        # Use model body to get encoder representations
        # the only ones we need for now are input_ids and attention_mask
        outputs = self.roberta(input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids, **kwargs)
        
        hidden_states = outputs['hidden_states'] # tuples with len 13 (number of layer/block)
        # each with shape: (bs,seq_len,hidden_size_len), e.g. for phobert: (bs,256, 768)
        # Note: hidden_size_len = embedding_size
        
        hidden_concat = torch.cat([hidden_states[i][:,0] for i in range(-1,-4-1,-1)],
                                  -1) # (bs,768*4)
        
        # classification head
        logits = self.classification_head(hidden_concat)

        loss = None
        if labels is not None:
            # labels shape: (bs,2), first is L3, second is L4
            labels_l1 = labels[:,0].view(-1) #(bs,)
            labels_l2 = labels[:,1].view(-1) #(bs,)
            l1_1hot = torch.nn.functional.one_hot(labels_l1, num_classes=self.size_l1)
            l2_1hot = torch.nn.functional.one_hot(labels_l2, num_classes=self.size_l2)
            label_concat_1hot = torch.cat((l1_1hot,l2_1hot),1) # (bs,L1+L2)

            # version 2: the original approach: positives and other children of same parents
            _mask = self.standard_mask[labels_l1].to(self.device)
            loss_func = torch.nn.BCEWithLogitsLoss(reduction='none')

            loss = loss_func(logits,label_concat_1hot.float())
            loss = torch.mul(loss,_mask)
            loss = (loss.sum(axis=1)/_mask.sum(axis=1)).mean()
            
        # Return model output object
        return SequenceClassifierOutput(loss=loss, logits=logits,
                                     hidden_states=None,
                                     attentions=outputs.attentions)

In [ ]:
show_doc(RobertaHSCCProbSequenceClassification)

---

[source](https://github.com/anhquan0412/that-nlp-library/blob/main/that_nlp_library/models/conditional_prob_classifiers.py#L46){target="_blank" style="float:right; font-size:smaller"}

### RobertaHSCCProbSequenceClassification

>      RobertaHSCCProbSequenceClassification (config, pretrained_roberta=None,
>                                             concathead_class=<class 'that_nlp_
>                                             library.models.classifiers.Roberta
>                                             ConcatHeadSimple'>,
>                                             classifier_dropout=0.1,
>                                             last_hidden_size=768,
>                                             size_l1=None, size_l2=None,
>                                             standard_mask=None, device=None)

Roberta Conditional Probability Architecture with Hidden-State-Concatenation for Sequence Classification task

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| config |  |  | HuggingFace model configuration |
| pretrained_roberta | NoneType | None | HuggingFace Roberta Body (useful for knowledge transfering task) |
| concathead_class | type | RobertaConcatHeadSimple |  |
| classifier_dropout | float | 0.1 | Dropout ratio (for dropout layer right before the last nn.Linear) |
| last_hidden_size | int | 768 | Last hidden size (before the last nn.Linear) |
| size_l1 | NoneType | None | Number of classes for head 1 |
| size_l2 | NoneType | None | Number of classes for head 2 |
| standard_mask | NoneType | None | Mask for conditional probability |
| device | NoneType | None | CPU or GPU |

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()